# Phase 3: Data Cleaning and Preparation

**Project:** Bank Customer Churn Analysis

**Goal:** create a clean, analysis-ready copy of the dataset while keeping all 10,000 customers.

**What this notebook does**
1. Loads the original CSV and confirms the file is unchanged.
2. Runs data quality checks.
3. Prepares the data, changing only what is necessary.
4. Adds five customer group columns.
5. Saves the cleaned data as a new file.
6. Validates the saved file.

**What this notebook does not do:** overwrite the original CSV, remove customers, fill in values, or analyse churn. Churn analysis starts in Phase 4.

Run this notebook from the `notebooks` folder, because the file paths below start from there.

## Step 1: Load and inspect the original dataset

A **SHA-256 fingerprint** is a code that changes if even one character of a file changes. We compare it with the value recorded in Phases 1 and 2 to prove that the original file is still the same.

In [1]:
import hashlib
import os

import pandas as pd

ORIGINAL_CSV = "../Bank Customer Churn Prediction.csv"
CLEANED_CSV = "../data/processed/bank_churn_cleaned.csv"
RECORDED_FINGERPRINT = "90f6625c31bdfcb0cde3b94c4f3004e43795e382921d0a5be67c4bd490b38818"


def get_file_fingerprint(file_path):
    with open(file_path, "rb") as file:
        return hashlib.sha256(file.read()).hexdigest()


print("Original file unchanged so far:", get_file_fingerprint(ORIGINAL_CSV) == RECORDED_FINGERPRINT)

Original file unchanged so far: True


In [2]:
original = pd.read_csv(ORIGINAL_CSV)

print("Rows:", original.shape[0])
print("Columns:", original.shape[1])
print()
print(original.dtypes)

Rows: 10000
Columns: 12

customer_id           int64
credit_score          int64
country              object
gender               object
age                   int64
tenure                int64
balance             float64
products_number       int64
credit_card           int64
active_member         int64
estimated_salary    float64
churn                 int64
dtype: object


In [3]:
original.head()

,customer_id,credit_score,country,gender,age,tenure,balance,products_number,credit_card,active_member,estimated_salary,churn
0,15634602,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,15647311,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,15619304,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,15701354,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,15737888,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


## Step 2: Data quality checks

We repeat the key checks here so that this notebook proves the data is fit for cleaning on its own, without relying on Phase 2.

### Missing values and duplicates

In [4]:
print("Empty cells:", original.isnull().sum().sum())
print("Duplicate rows:", original.duplicated().sum())
print("Duplicate customer IDs:", original["customer_id"].duplicated().sum())

Empty cells: 0
Duplicate rows: 0
Duplicate customer IDs: 0


### Text values and binary columns

Text columns should contain only the expected spellings. Binary columns should contain only 0 and 1.

In [5]:
print("Countries:", sorted(original["country"].unique()))
print("Genders:", sorted(original["gender"].unique()))
print()

for column in ["credit_card", "active_member", "churn"]:
    print(column, "values:", sorted(original[column].unique()))

Countries:

 ['France', 'Germany', 'Spain']
Genders: ['Female', 'Male']

credit_card values: [0, 1]
active_member values: [0, 1]
churn values: [0, 1]


### Numerical ranges and invalid values

Each line counts the rows that break a reasonableness rule, so **0 is the result we want**. These are the same rules used in Phase 2.

In [6]:
original[["credit_score", "age", "tenure", "balance", "products_number", "estimated_salary"]].agg(["min", "max"]).T

,min,max
credit_score,350.00,850.00
age,18.00,92.00
tenure,0.00,10.00
balance,0.00,250898.09
products_number,1.00,4.00
estimated_salary,11.58,199992.48


In [7]:
print("credit_score outside 300-850:", ((original["credit_score"] < 300) | (original["credit_score"] > 850)).sum())
print("age outside 18-100:", ((original["age"] < 18) | (original["age"] > 100)).sum())
print("negative tenure:", (original["tenure"] < 0).sum())
print("tenure not lower than age:", (original["tenure"] >= original["age"]).sum())
print("negative balance:", (original["balance"] < 0).sum())
print("estimated_salary of zero or below:", (original["estimated_salary"] <= 0).sum())
print("products_number outside 1-4:", ((original["products_number"] < 1) | (original["products_number"] > 4)).sum())

credit_score outside 300-850: 0
age outside 18-100: 0
negative tenure: 0
tenure not lower than age: 0
negative balance: 0
estimated_salary of zero or below: 0
products_number outside 1-4: 0


### A closer look at very low salaries

Phase 2 flagged some very low `estimated_salary` values for a closer look. They are unusual, but nothing in the data proves they are wrong. The column is an *estimate*, and its pay period and currency are unknown.

**Decision:** keep these customers and their values unchanged. Removing them would mean losing valid customer records, and replacing the values would mean inventing data.

In [8]:
print("estimated_salary under 1,000:", (original["estimated_salary"] < 1000).sum())
print("estimated_salary under 100:", (original["estimated_salary"] < 100).sum())

original.sort_values("estimated_salary")[["customer_id", "age", "balance", "estimated_salary"]].head(5)

estimated_salary under 1,000: 59
estimated_salary under 100: 4


,customer_id,age,balance,estimated_salary
2362,15791053,45,122917.71,11.58
9647,15679693,31,0.00,90.07
9010,15786463,59,121669.93,91.75
5548,15639662,38,0.00,96.27
4989,15602851,40,0.00,106.67


## Step 3: Prepare the data

We work on a **copy**, so the `original` table stays available for comparison later.

### Remove unnecessary whitespace

`.str.strip()` removes spaces from the start and end of text. We count the changes so that nothing changes silently.

In [9]:
cleaned = original.copy()

for column in ["country", "gender"]:
    cleaned[column] = cleaned[column].str.strip()
    values_changed = (cleaned[column] != original[column]).sum()
    print(column, "values changed by whitespace removal:", values_changed)

country values changed by whitespace removal: 0
gender values changed by whitespace removal: 0


### Data types

Each column already has a suitable type, so **no type changes are needed**. Changing them anyway would add risk with no benefit. `credit_card`, `active_member` and `churn` stay as 0 and 1 because that makes them easy to add up in SQL and Power BI.

## Step 4: Create the customer group columns

`pd.cut()` places each number into a labelled group:

- `bins` lists the edges of the groups.
- `labels` names the groups.
- `right=False` means an edge value belongs to the **higher** group. For example, age 30 goes into "30 to 39".

The original numerical columns are not overwritten. Each group is a **new** column.

**About the labels:** we write "0 to 2" rather than "0-2" because spreadsheet programs can turn text such as "3-5" into a date.

In [10]:
no_upper_limit = float("inf")  # "inf" means infinity, so the last group has no upper limit

cleaned["age_group"] = pd.cut(
    cleaned["age"],
    bins=[0, 30, 40, 50, 60, no_upper_limit],
    labels=["Under 30", "30 to 39", "40 to 49", "50 to 59", "60 and above"],
    right=False,
)

cleaned["tenure_group"] = pd.cut(
    cleaned["tenure"],
    bins=[0, 3, 6, 9, no_upper_limit],
    labels=["0 to 2", "3 to 5", "6 to 8", "9 to 10"],
    right=False,
)

cleaned["credit_score_band"] = pd.cut(
    cleaned["credit_score"],
    bins=[0, 580, 670, 740, 800, no_upper_limit],
    labels=["Below 580", "580 to 669", "670 to 739", "740 to 799", "800 and above"],
    right=False,
)

cleaned["salary_band"] = pd.cut(
    cleaned["estimated_salary"],
    bins=[0, 50000, 100000, 150000, no_upper_limit],
    labels=["Under 50,000", "50,000 to under 100,000", "100,000 to under 150,000", "150,000 and above"],
    right=False,
)

### Balance bands

Balance is handled differently, because customers with exactly zero need their own group.

- Here we leave out `right=False`, so each group **includes its upper edge**. A balance of exactly 100,000 would go into "Above 0 to 100,000".
- The first group runs from just above -1 up to and including 0. Step 2 confirmed there are no negative balances, so it holds only zero balances.
- As approved, the small 75-customer "above 0 to 50,000" group is merged with the next group to form "Above 0 to 100,000".

In [11]:
cleaned["balance_band"] = pd.cut(
    cleaned["balance"],
    bins=[-1, 0, 100000, 150000, no_upper_limit],
    labels=["Zero", "Above 0 to 100,000", "Above 100,000 to 150,000", "Above 150,000"],
)

### Check the size and actual range of every group

This table shows how many customers are in each group, and the lowest and highest actual values inside it. Churn is deliberately not used here.

In [12]:
group_columns = {
    "age_group": "age",
    "tenure_group": "tenure",
    "credit_score_band": "credit_score",
    "salary_band": "estimated_salary",
    "balance_band": "balance",
}

for group_column, source_column in group_columns.items():
    print(group_column)
    print(cleaned.groupby(group_column, observed=False)[source_column].agg(["count", "min", "max"]))
    print()

age_group
              count  min  max
age_group                    
Under 30       1641   18   29
30 to 39       4346   30   39
40 to 49       2618   40   49
50 to 59        869   50   59
60 and above    526   60   92

tenure_group
              count  min  max
tenure_group                 
0 to 2         2496    0    2
3 to 5         3010    3    5
6 to 8         3020    6    8
9 to 10        1474    9   10

credit_score_band
                   count  min  max
credit_score_band                 
Below 580           2362  350  579
580 to 669          3331  580  669
670 to 739          2428  670  739
740 to 799          1224  740  799
800 and above        655  800  850

salary_band
                          count        min        max
salary_band                                          
Under 50,000               2453      11.58   49978.67
50,000 to under 100,000    2537   50016.17   99984.86
100,000 to under 150,000   2555  100015.79  149991.32
150,000 and above          2455  150026

## Step 5: Save the cleaned dataset

- `index=False` stops pandas from adding an extra row-number column.
- `lineterminator="\n"` keeps the same line endings as the original file.
- We save to a **new file** in `data/processed`, so the original CSV is not touched.

In [13]:
os.makedirs("../data/processed", exist_ok=True)

cleaned.to_csv(CLEANED_CSV, index=False, lineterminator="\n")

print("Saved:", CLEANED_CSV)
print("File size (bytes):", os.path.getsize(CLEANED_CSV))

Saved: ../data/processed/bank_churn_cleaned.csv
File size (bytes): 1223569


## Step 6: Validate the saved file

We reload the saved file from disk instead of trusting the table in memory, because the file is what later phases will use.

Each check prints PASS or FAIL. The expected group counts come from Phase 2, and the merged balance group is 75 + 1,509 = 1,584.

In [14]:
saved = pd.read_csv(CLEANED_CSV)

results = []


def check(description, passed):
    results.append(passed)
    print("PASS" if passed else "FAIL", "-", description)


original_columns = list(original.columns)

expected_groups = {
    "age_group": {"Under 30": 1641, "30 to 39": 4346, "40 to 49": 2618, "50 to 59": 869, "60 and above": 526},
    "tenure_group": {"0 to 2": 2496, "3 to 5": 3010, "6 to 8": 3020, "9 to 10": 1474},
    "credit_score_band": {"Below 580": 2362, "580 to 669": 3331, "670 to 739": 2428, "740 to 799": 1224, "800 and above": 655},
    "salary_band": {"Under 50,000": 2453, "50,000 to under 100,000": 2537, "100,000 to under 150,000": 2555, "150,000 and above": 2455},
    "balance_band": {"Zero": 3617, "Above 0 to 100,000": 1584, "Above 100,000 to 150,000": 3830, "Above 150,000": 969},
}

check("Customer count matches the original (10,000)", len(saved) == len(original) == 10000)
check("The file has 17 columns (12 original + 5 new)", saved.shape[1] == 17)
check("Original columns are present, in the same order", list(saved.columns[:12]) == original_columns)
check("Customer IDs are unique", saved["customer_id"].is_unique)
check("Customer IDs are in the same order as the original", saved["customer_id"].equals(original["customer_id"]))
check("No missing values anywhere", saved.isnull().sum().sum() == 0)

for column in original_columns:
    check(f"{column} is unchanged from the original", saved[column].equals(original[column]))

for column, expected_counts in expected_groups.items():
    actual_counts = saved[column].value_counts().to_dict()
    check(f"{column} has the expected groups and customer counts", actual_counts == expected_counts)

check("Original CSV fingerprint is unchanged", get_file_fingerprint(ORIGINAL_CSV) == RECORDED_FINGERPRINT)

print()
print("Checks passed:", sum(results), "of", len(results))

PASS - Customer count matches the original (10,000)
PASS - The file has 17 columns (12 original + 5 new)
PASS - Original columns are present, in the same order
PASS - Customer IDs are unique
PASS - Customer IDs are in the same order as the original
PASS - No missing values anywhere
PASS - customer_id is unchanged from the original
PASS - credit_score is unchanged from the original
PASS - country is unchanged from the original
PASS - gender is unchanged from the original
PASS - age is unchanged from the original
PASS - tenure is unchanged from the original
PASS - balance is unchanged from the original
PASS - products_number is unchanged from the original
PASS - credit_card is unchanged from the original
PASS - active_member is unchanged from the original
PASS - estimated_salary is unchanged from the original
PASS - churn is unchanged from the original
PASS - age_group has the expected groups and customer counts
PASS - tenure_group has the expected groups and customer counts
PASS - credi

In [15]:
saved.head()

,customer_id,credit_score,country,gender,age,tenure,balance,products_number,credit_card,active_member,estimated_salary,churn,age_group,tenure_group,credit_score_band,salary_band,balance_band
0,15634602,619,France,Female,42,2,0.00,1,1,1,101348.88,1,40 to 49,0 to 2,580 to 669,"100,000 to under 150,000",Zero
1,15647311,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0,40 to 49,0 to 2,580 to 669,"100,000 to under 150,000","Above 0 to 100,000"
2,15619304,502,France,Female,42,8,159660.80,3,1,0,113931.57,1,40 to 49,6 to 8,Below 580,"100,000 to under 150,000","Above 150,000"
3,15701354,699,France,Female,39,1,0.00,2,0,0,93826.63,0,30 to 39,0 to 2,670 to 739,"50,000 to under 100,000",Zero
4,15737888,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0,40 to 49,0 to 2,800 and above,"50,000 to under 100,000","Above 100,000 to 150,000"
